# Capstone Project: Agentic RAG-Based AI Research Assistant
## Natural Language Processing (NLP) — Practical Coursework & Viva Demonstration

### Project Goal
The primary objective of this project is to develop an autonomous, locally hosted AI Research Assistant that overcomes the fundamental limitations of modern Large Language Models (LLMs)—specifically factual hallucination, knowledge cutoff, and lack of dynamic tool execution. By combining **Retrieval-Augmented Generation (RAG)**, **Vector Databases (ChromaDB)**, **ReAct Agents**, and custom **NLP Tools** running on a **Local LLM (Ollama - LLaMA 3)**, this system operates 100% offline with zero cloud API costs.

---

### Professor's Mandatory Key Topics Implemented
1. **RAG (Retrieval-Augmented Generation):** Eliminates factual hallucination by grounding LLM responses in real indexed paper excerpts.
2. **Agents:** Implements the ReAct (Reasoning + Acting) autonomous architecture that reasons dynamically before responding.
3. **Tools:** Connects specialized capabilities (Semantic Document Search, Sentiment Analysis, Named Entity Recognition, and Multilingual Translation).
4. **Vector DB (ChromaDB):** Indexes dense 384-dimensional vector embeddings for sub-millisecond similarity retrieval.
5. **Local LLM (Ollama - LLaMA 3):** Powers generation and reasoning locally on hardware with no external data transmission.

---

### Story for the Presentation
In the modern academic landscape, scientific literature is expanding exponentially. Researchers are overwhelmed by hundreds of complex publications, domain-specific terminology, foreign language papers, and subjective peer reviewer remarks. 

Our **Agentic RAG Assistant** acts as an autonomous copilot:
- When asked a factual question, it queries the **Vector DB** using semantic search (**RAG**).
- When asked about reviewer tone, it invokes a **Sentiment Analysis Tool**.
- When asked to identify researchers or institutions, it invokes an **Information Extraction Tool (spaCy NER)**.
- When presented with foreign language inputs, it invokes a **Multilingual Machine Translation Tool**.
- The **ReAct Agent** makes all tool invocation decisions autonomously using **Prompt Engineering**.

---

### Syllabus & Course Outcome (CO) Mapping (Practicals 1 to 7)

| Practical / Step | Syllabus Module | Course Outcome | Core Technology & Implementation |
| :--- | :--- | :--- | :--- |
| **Practical 1** | Text Extraction & Preprocessing | **CO1** | `pypdf`, `RecursiveCharacterTextSplitter` |
| **Practical 1** | Vectorization & Vector Database | **CO1** | `sentence-transformers/all-MiniLM-L6-v2`, `ChromaDB` (RAG Core) |
| **Practical 2** | Sentiment & Subjectivity Analysis | **CO2** | `TextBlob` Lexical Polarity & Subjectivity |
| **Practical 3** | Information Extraction (NER) | **CO2** | `spaCy` (`en_core_web_sm`) Token Classification |
| **Practical 4** | Unsupervised Topic Modeling | **CO3** | `BERTopic` with c-TF-IDF & Embeddings |
| **Practical 5** | Multilingual NLP & Machine Translation | **CO3** | `deep-translator` (GoogleTranslator) |
| **Practical 6** | Agentic Workflow & Tool Use | **CO4** | LangChain `AgentExecutor`, Multi-Tool Orchestration |
| **Practical 7** | Generative AI & Prompt Engineering | **CO4** | `Ollama` (`llama3`), ReAct Prompt Template (`hwchase17/react`) |

In [ ]:
# Cell 2: Library Imports and Setup
import os
import re
import pandas as pd
import numpy as np

# Document Ingestion & Text Splitting
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Database & Dense Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Classical NLP: Sentiment Analysis & Information Extraction
from textblob import TextBlob
import spacy

# Topic Modeling & Multilingual NLP
from bertopic import BERTopic
from deep_translator import GoogleTranslator

# Local LLM (Ollama), Tools, and ReAct Agent Framework
from langchain_ollama import ChatOllama
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

# Robust hub import
try:
    from langchainhub import hub
except ImportError:
    try:
        from langchain import hub
    except ImportError:
        hub = None

try:
    from langchain.agents import AgentExecutor, create_react_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_react_agent

# Load spaCy pipeline for Information Extraction (NER)
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("All core NLP libraries, spaCy model, and local Ollama interfaces loaded successfully!")

## 1. Text Extraction & Preprocessing (Practical 1 - CO1)

### Theoretical Background
Academic papers typically exist in binary PDF format. Before conducting vectorization or semantic analysis, the raw binary stream must be parsed and partitioned:

1. **Text Extraction:** Using `pypdf`, we iterate through each page object, extract clean textual streams, and normalize whitespace.
2. **Recursive Character Chunking:** Large documents cannot be embedded in one pass because transformer embedding models have finite token context windows (e.g., 256 or 512 tokens). Splitting text by a rigid character count truncates sentences midway. Instead, **`RecursiveCharacterTextSplitter`** recursively evaluates paragraph breaks (`\n\n`), newline boundaries (`\n`), sentence ends (`. `), and spaces (` `).
3. **Sliding Overlap:** An overlap of 100 characters ensures semantic continuity between adjacent chunks, preventing critical context from being severed at chunk boundaries.

In [ ]:
# Cell 4: Text Extraction and Chunking Function
def extract_and_chunk_pdf(pdf_path=None, chunk_size=1000, overlap=100):
    """
    Extracts text from a given PDF file or loads educational demonstration corpus,
    then segments the text into overlapping syntactic chunks.
    """
    extracted_text = ""
    
    if pdf_path and os.path.exists(pdf_path):
        reader = PdfReader(pdf_path)
        pages_text = []
        for page in reader.pages:
            txt = page.extract_text()
            if txt:
                pages_text.append(re.sub(r'\s+', ' ', txt).strip())
        extracted_text = "\n\n".join(pages_text)
        print(f"Extracted text from '{pdf_path}' ({len(extracted_text)} characters).")
    else:
        print("No external PDF file found. Loading sample academic research corpus...")
        extracted_text = """
Title: Autonomous Agentic RAG Frameworks in Modern Natural Language Processing.
Authors: Dr. Emily Watson, Dr. Rajesh Kumar, and Stanford Artificial Intelligence Laboratory.
Publication: International Conference on Machine Learning, 2024.

Abstract: Transformer architectures have revolutionized Natural Language Processing by replacing recurrent architectures with multi-head self-attention mechanisms. However, Large Language Models (LLMs) frequently suffer from factual hallucinations when answering queries outside their static training weights. In this research, we introduce an autonomous agentic framework combining local LLMs with dense vector retrieval. We demonstrate how Retrieval-Augmented Generation (RAG) grounds generated outputs in verified textual corpora, reducing factual hallucination by over 92%.

Section 1: Introduction and Literature Review. Traditional sequence models suffered from vanishing gradients across long textual dependencies. Vaswani et al. introduced the Transformer, which enables parallelized computation across massive multilingual datasets. Despite their fluency, generative LLMs lack internal fact verification mechanisms. When prompted with nuanced scientific inquiries, they hallucinate non-existent authors, datasets, and benchmark scores.

Section 2: Methodology and Dense Vector Representation. Our pipeline ingests research literature, partitions documents into 1000-character segments with 100-character overlap, and projects passages into a 384-dimensional dense vector space using HuggingFace sentence-transformers (all-MiniLM-L6-v2). All embeddings are indexed inside a local ChromaDB instance using Hierarchical Navigable Small World (HNSW) graphs.

Section 3: Empirical Evaluation and Sentiment Profiling. Scientific peer reviewers evaluated our framework across 500 academic benchmarks. Sentiment analysis of reviewer commentary indicates strongly positive polarity toward explainable agent trajectories. Furthermore, token classification models successfully extract critical named entities including author identities, research labs, and numerical metrics.

Section 4: Conclusion. Operating completely offline with local Ollama LLaMA 3 guarantees complete institutional data privacy, zero API costs, and robust academic research synthesis.
""".strip()

    # Recursive character text splitting for Vector DB preparation
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "]
    )
    
    chunks = splitter.create_documents([extracted_text])
    return extracted_text, chunks

# Execute Demonstration
raw_text, document_chunks = extract_and_chunk_pdf("sample_paper.pdf")
print(f"\nTotal Document Chunks Produced: {len(document_chunks)}")
for i, chunk in enumerate(document_chunks, 1):
    print(f"--- Chunk {i} (Length: {len(chunk.page_content)} characters) ---")
    print(chunk.page_content[:180] + "...\n")

## 2. Vectorization & Vector DB (Practical 1 - CO1)

### Theoretical Background: The Foundation of RAG
To enable semantic retrieval, text must be mapped into continuous numeric vector spaces:

1. **Dense Vector Embeddings:** Unlike sparse representations (TF-IDF or Bag-of-Words) which only match exact keywords, dense sentence transformer models (`sentence-transformers/all-MiniLM-L6-v2`) encode semantic concepts into **384-dimensional dense vectors**. Semantically synonymous sentences (e.g., *"transformer attention mechanisms"* and *"neural sequence modeling"*) map close together in vector space.
2. **ChromaDB (Vector Database):** Stores the embeddings and indexes them using **HNSW (Hierarchical Navigable Small World)** graphs. When a query is submitted, ChromaDB computes the Cosine Similarity between the query vector and document chunk vectors, retrieving the top-$k$ nearest passages in milliseconds.

$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

This forms the critical **Retrieval** component in our **RAG** architecture.

In [ ]:
# Cell 6: Embedding Generation and Vector Store Retrieval (RAG Foundation)
print("Initializing HuggingFace embedding model (sentence-transformers/all-MiniLM-L6-v2)...")
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Create local in-memory Chroma vector database
vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_function,
    collection_name="research_papers"
)
print("ChromaDB vector store initialized and indexed!")

# Demonstration of Semantic Information Retrieval (The 'R' in RAG)
test_query = "How does the methodology eliminate factual hallucination?"
print(f"\nExecuting Similarity Search for Query: '{test_query}'")
retrieved_results = vectorstore.similarity_search(test_query, k=2)

for idx, doc in enumerate(retrieved_results, 1):
    print(f"\n[Top Match {idx}]:")
    print(doc.page_content.strip())

## 3. Classic NLP Tasks: Sentiment & Info Extraction (Practical 2 & 3 - CO2)

### Theoretical Background
A comprehensive research assistant must perform structured linguistic analysis alongside text generation:

1. **Sentiment Analysis (Practical 2):** Using **TextBlob**, we compute two key metrics:
   - **Polarity:** Measures emotional valence from $-1.0$ (strongly negative) to $+1.0$ (strongly positive).
   - **Subjectivity:** Measures whether text is objective empirical fact ($0.0$) versus personal opinion ($1.0$).
2. **Information Extraction / NER (Practical 3):** Using **spaCy** token classification models (`en_core_web_sm`), we extract structured Named Entities including:
   - `PERSON`: Author names.
   - `ORG`: Universities, research labs, companies.
   - `DATE`: Publication years, timeline dates.
   - `GPE`: Geopolitical entities and countries.

In [ ]:
# Cell 8: Sentiment Analysis & Named Entity Recognition Functions

# Task 1: Sentiment Analysis with TextBlob
def analyze_sentiment(text: str):
    """
    Analyzes sentiment polarity and subjectivity using TextBlob.
    """
    blob = TextBlob(text)
    polarity = round(blob.sentiment.polarity, 4)
    subjectivity = round(blob.sentiment.subjectivity, 4)
    
    if polarity > 0.1:
        category = "Positive"
    elif polarity < -0.1:
        category = "Negative"
    else:
        category = "Neutral"
        
    return {
        "Polarity": polarity,
        "Subjectivity": subjectivity,
        "Category": category
    }

# Task 2: Information Extraction (NER) with spaCy
def extract_named_entities(text: str):
    """
    Extracts Named Entities (PERSON, ORG, DATE, GPE, etc.) using spaCy.
    """
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        entities.append({"Entity": ent.text, "Label": ent.label_, "Explanation": spacy.explain(ent.label_)})
    return entities

# Demonstration on sample research statements
sample_review = "The paper presents exceptional results with outstanding accuracy, but traditional RNNs were catastrophic."
sentiment_res = analyze_sentiment(sample_review)
print("--- 1. Sentiment Analysis Demonstration ---")
print(f"Text: '{sample_review}'")
print(f"Result: {sentiment_res['Category']} (Polarity: {sentiment_res['Polarity']}, Subjectivity: {sentiment_res['Subjectivity']})\n")

sample_sentence = "Dr. Emily Watson and Dr. Rajesh Kumar presented their research at Stanford University in 2024."
ner_res = extract_named_entities(sample_sentence)
print("--- 2. Information Extraction (NER) Demonstration ---")
print(f"Text: '{sample_sentence}'")
ner_df = pd.DataFrame(ner_res)
display(ner_df)

## 4. Topic Modeling (Practical 4 - CO3)

### Theoretical Background
Topic modeling enables unsupervised thematic discovery across large collections of unlabelled academic literature. Unlike classical Latent Dirichlet Allocation (LDA) which relies purely on bag-of-words word frequencies, **BERTopic** combines:
1. Dense sentence transformer embeddings.
2. UMAP dimensionality reduction.
3. HDBSCAN density clustering.
4. Class-based TF-IDF (**c-TF-IDF**) to extract the most descriptive keyword tokens per cluster.

In [ ]:
# Cell 10: Topic Modeling Demonstration with BERTopic
mock_corpus = [
    "Transformers, attention mechanisms, and self-attention revolutionize sequence learning in neural networks.",
    "Large language models and decoder architectures excel at generative summarization and contextual understanding.",
    "Vector databases like ChromaDB store dense geometric embeddings for similarity search.",
    "Approximate nearest neighbors and cosine distance accelerate document retrieval in vector stores.",
    "Named entity recognition and token classification identify persons, organizations, and dates in text.",
    "Information extraction pipelines parse unstructured medical and legal documents into relational databases."
]

print(f"Fitting BERTopic model on {len(mock_corpus)} representative documents...")
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", min_topic_size=2, verbose=False)
topics, probs = topic_model.fit_transform(mock_corpus)

topic_info = topic_model.get_topic_info()
print("\n--- Unsupervised Topic Clusters Extracted ---")
display(topic_info[["Topic", "Count", "Name"]].head())

## 5. Multilingual NLP (Practical 5 - CO3)

### Theoretical Background
Scientific research is global. A multilingual research assistant must understand queries in different languages and translate findings accurately. Using **deep-translator**, we implement cross-lingual translation between non-English inputs (e.g., French, Marathi, Hindi, Spanish) and English.

In [ ]:
# Cell 12: Multilingual Machine Translation
def translate_to_english(text: str, source_lang="auto"):
    """
    Translates non-English inputs into English using deep-translator.
    """
    translator = GoogleTranslator(source=source_lang, target="en")
    return translator.translate(text)

# Test strings: French & Marathi
french_text = "L'intelligence artificielle transforme la recherche scientifique et le traitement du langage naturel."
marathi_text = "कृत्रिम बुद्धिमत्ता नैसर्गिक भाषा प्रक्रियेमध्ये नवीन क्रांती घडवून आणत आहे."

print("--- Multilingual Translation Demonstration ---")
print("[French Input]:", french_text)
print("[Translated to English]:", translate_to_english(french_text, source_lang="fr"))

print("\n[Marathi Input]:", marathi_text)
print("[Translated to English]:", translate_to_english(marathi_text, source_lang="mr"))

## 6. Agentic RAG & Prompt Engineering with Local LLM (Practical 6 - CO4)

### Theoretical Background: Agents, Tools, and ReAct Framework
In conventional RAG pipelines, execution is static and hardcoded: `User Query -> Vector Search -> Context Injection -> LLM Answer`. 

In an **Agentic RAG** system, the **Local LLM (Ollama - LLaMA 3)** is given agency over execution. It acts as the central reasoning engine (**Agent**) equipped with a toolbox of discrete capabilities (**Tools**):
- `Document_Search`: The RAG retrieval tool querying ChromaDB.
- `Sentiment_Analysis`: Evaluates reviewer or author tone via TextBlob.
- `Info_Extraction`: Parses author/organization entities via spaCy NER.
- `Translate`: Overcomes language barriers via deep-translator.

### Prompt Engineering: The ReAct Protocol
Through **Prompt Engineering** (`hwchase17/react`), the agent follows the structured **Reasoning + Acting** cycle:
$$\text{Question} \longrightarrow \text{Thought} \longrightarrow \text{Action (Tool Selection)} \longrightarrow \text{Action Input} \longrightarrow \text{Observation} \dots \longrightarrow \text{Final Answer}$$

This seamlessly combines **LLM**, **Agents**, **Tools**, and **RAG** into one unified, offline system.

In [ ]:
# Cell 14: Agentic RAG & Prompt Engineering Setup with Local Ollama (llama3)

# 1. Initialize Local Ollama LLM (llama3)
print("Connecting to Local Ollama instance (model='llama3', temperature=0)...")
llm = ChatOllama(
    model="llama3",
    temperature=0.0
)

# 2. Wrap our NLP Functions into LangChain Tools
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def tool_doc_search(query: str) -> str:
    docs = retriever.invoke(query)
    if not docs:
        return "No directly matching excerpt found in the document."
    return "\n\n".join([f"[Excerpt {i+1}]: {d.page_content}" for i, d in enumerate(docs)])

def tool_sentiment(text: str) -> str:
    res = analyze_sentiment(text)
    return f"Sentiment: {res['Category']} (Polarity: {res['Polarity']}, Subjectivity: {res['Subjectivity']})"

def tool_info_extraction(text: str) -> str:
    entities = extract_named_entities(text)
    if not entities:
        return "No named entities detected."
    return "; ".join([f"{e['Entity']} ({e['Label']})" for e in entities])

def tool_translate(text: str) -> str:
    return translate_to_english(text)

tools = [
    Tool(
        name="Document_Search",
        func=tool_doc_search,
        description="Search facts, methodologies, benchmarks, and data inside the uploaded research paper."
    ),
    Tool(
        name="Sentiment_Analysis",
        func=tool_sentiment,
        description="Analyze the emotional polarity and tone of any passage or statement."
    ),
    Tool(
        name="Info_Extraction",
        func=tool_info_extraction,
        description="Extract key named entities (Persons, Organizations, Universities, Dates) from text."
    ),
    Tool(
        name="Translate",
        func=tool_translate,
        description="Translate non-English text queries or passages into English."
    )
]

# 3. Prompt Engineering: Pull ReAct prompt or use canonical offline template
react_prompt = None
if hub is not None:
    try:
        react_prompt = hub.pull("hwchase17/react")
    except Exception:
        react_prompt = None

if react_prompt is None:
    template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format strictly:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

CRITICAL RULES:
1. When an Observation answers the question, write 'Thought: I now know the final answer' and provide 'Final Answer: ...'.
2. Never repeat identical tool calls with the same input.
3. Never invent tool names outside of [{tool_names}].

Begin!

Question: {input}
Thought:{agent_scratchpad}"""
    react_prompt = PromptTemplate.from_template(template)

# 4. Construct ReAct Agent & AgentExecutor
agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    max_execution_time=120,
    early_stopping_method="generate"
)

# 5. Run Live Demonstration Query
test_prompt = "Search the document to explain how the authors solve hallucination, and state the sentiment of their findings."
print(f"--- Executing Agent Query: '{test_prompt}' ---\n")
agent_result = agent_executor.invoke({"input": test_prompt})

print("\n================ AGENT FINAL ANSWER ================")
print(agent_result["output"])